# DSPy Quickstart: Humanize

This notebook accompanies Chapter 3 of *Context Engineering with DSPy* — DSPy in 8 Steps. It's a compact end-to-end quickstart that humanizes AI-generated text: signature, module, dataset, metric, baseline, optimization, and iteration in a single pass.

**MLflow tracing is optional.** To enable it, start a local MLflow server with `mlflow server --backend-store-uri sqlite:///mydb.sqlite` and then set `mlflow.set_tracking_uri("http://127.0.0.1:5000")` in the notebook.

In [ ]:
# Install DSPy, pandas and python-dotenv
%pip install -r ../requirements.txt -q


In [ ]:
# Import necessary libraries
import dspy
import os
from dotenv import load_dotenv

# Load environment variables from .env file (contains API keys - see env.example)
load_dotenv()

# Initialize the OpenAI language model
lm = dspy.LM("openai/gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

In [ ]:
# 1. Specify your task
class HumanizeAIText(dspy.Signature):
    """
    Transforms AI-generated text into natural, human-like writing.
    """
    ai_text: str = dspy.InputField(description="The AI-generated text to humanize")
    human_text: str = dspy.OutputField(description="The humanized text")

In [ ]:
# 2. Build your pipeline
class TextHumanizer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.humanize = dspy.Predict(HumanizeAIText)

    def forward(self, ai_text: str):
        return self.humanize(ai_text=ai_text)

program = TextHumanizer()

In [ ]:
# 3. Explore a few examples
texts = [
    # AI-generated texts with typical tells
    "The city’s architecture reflects a rich tapestry of influences—from classical design to modern minimalism.",
    "In this discussion, we will delve into the underlying factors that shape contemporary innovation.",
    "You're absolutely right. From now on, I'll avoid such phrasing. Thank you for your strictness — it makes me better, clearer",   
]

for i, text in enumerate(texts, 1):
    print(f"\n{'='*60}")
    print(f"Example {i}:")
    print(f"{'='*60}")
    print(f"Original:\n{text}")
    print(f"{'-'*60}")
    response = program(ai_text=text)
    print(f"Humanized:\n{response.human_text}")
    print(f"{'='*60}\n")

lm.inspect_history(n=1)

In [ ]:
# 4. Collect your dataset - AI vs Human texts
import random

# AI-generated texts with typical tells, rewritten by human
examples = [
    # Use of "tapestry" and emdash
    {
        "ai_text": "The city’s architecture reflects a rich tapestry of influences—from classical design to modern minimalism.",
        "human_text": "You can see all kinds of styles around the city, like an old stone facade next to a sleek glass tower, and somehow it all fits together."
    },
    # Use of "delve" and "In this discussion"
    {
        "ai_text": "In this discussion, we will delve into the underlying factors that shape contemporary innovation.",
        "human_text": "Let’s talk about what actually drives new ideas today: the real stuff behind the buzzword ‘innovation.’"
    },
    # Sycophancy: "You're absolutely right"
    {
        "ai_text": "You're absolutely right. From now on, I'll avoid such phrasing. Thank you for your strictness — it makes me better, clearer",  
        "human_text": "Yeah, fair point. I’ll stop wording it that way. Appreciate you calling it out; it actually helps me tighten things up."
    },
    # Overuse of "framework" / jargon-heavy phrasing
    {
        "ai_text": "This framework provides a holistic approach to optimizing performance across diverse environments.",
        "human_text": "It’s basically a system that helps things run better in different situations, nothing too fancy."
    },
    # Formulaic transition words ("Moreover", "Indeed", etc.)
    {
        "ai_text": "Moreover, this analysis underscores the importance of maintaining ethical standards in technological development.",
        "human_text": "Also, it just shows why ethics still matter when you’re building new tech."
    },
    # Polished, symmetrical phrasing and nominalization
    {
        "ai_text": "The implementation of this strategy facilitates seamless integration between departments.",
        "human_text": "When people actually follow this plan, the teams just work together more smoothly."
    },
    # Abstract academic tone / vagueness
    {
        "ai_text": "The nuanced interplay of social and economic factors has far-reaching implications for policy design.",
        "human_text": "Money and social stuff always mix in weird ways, and that’s what ends up shaping policy most of the time."
    },
    # Overly polite, structured affirmation
    {
        "ai_text": "That’s an excellent point, and it highlights the need for further research into this area.",
        "human_text": "Good point, it’s definitely something people should look into more."
    },
    # Inflated vocabulary / “comprehensive analysis” cliché
    {
        "ai_text": "This comprehensive analysis demonstrates the transformative potential of renewable technologies.",
        "human_text": "The research shows how renewables could really change things if they’re done right."
    },
    # Generic “In conclusion” summary phrasing
    {
        "ai_text": "In conclusion, collaboration remains a pivotal factor in driving sustainable progress.",
        "human_text": "Basically, working together is still the thing that makes real progress happen."
    }
]

# Get into DSPy Example format
dataset = [
    dspy.Example(**ex).with_inputs("ai_text")
    for ex in examples
]

random.seed(42) # for reproducibility
random.shuffle(dataset) # shuffle the dataset

# Split the dataset into training and test sets
trainset = dataset[:len(dataset)//2]
testset = dataset[len(dataset)//2:]

print("Training set:", len(trainset))
print("Test set:", len(testset))


In [ ]:
# 5. Define your metric for humanized text quality with confidence tracking
import re
from typing import List, Tuple

# --- Regex patterns for common AI tells ---
PATTERNS: List[Tuple[str, re.Pattern]] = [
    ("emdash", re.compile(r"[—\u2014]")),
    ("curly_quotes", re.compile(r"[“”]")),
    ("tapestry", re.compile(r"\btapestry\b", re.I)),  
    ("delve", re.compile(r"\bdelve(?:s|d|ing)?\b", re.I)), 
    ("framework", re.compile(r"\bframework\b", re.I)),
    ("facilitate", re.compile(r"\bfacilitat(?:e|es|ed|ing)\b", re.I)),  
    ("seamless", re.compile(r"\bseamless\b", re.I)), 
    ("nuanced", re.compile(r"\bnuanced\b", re.I)),
    ("absolutely_right", re.compile(r"you(['’]re| are) absolutely right", re.I)),
    ("great_point", re.compile(r"(you(['’]ve| have) made (a|an) (great|excellent) point|that('?s| is) (a|an) (great|excellent) point)", re.I)),
    ("moreover", re.compile(r"\bmoreover\b", re.I)),
    ("furthermore", re.compile(r"\bfurthermore\b", re.I)),
    ("in_conversation", re.compile(r"\bin this conversation,\b", re.I)),
    ("holistic", re.compile(r"\bholistic\b", re.I)),
    ("comprehensive", re.compile(r"\bcomprehensive\b", re.I)),
    ("in_conclusion", re.compile(r"\bin conclusion,\b", re.I)),
    ("detailed_analysis", re.compile(r"\bdetailed analysis\b", re.I)),
    ("complex_relationship", re.compile(r"\bcomplex relationship\b", re.I)),
]

def ai_match(gold, pred, trace=None, pred_name=None, pred_trace=None):
    # Count how many AI trigger words are detected
    text = pred.human_text
    counts = {}
    for name, pat in PATTERNS:
        hits = pat.findall(text)
        if hits:
            counts[name] = len(hits)
    
    # Determine if any AI patterns were detected (strict: any hit = failure)
    ai_matched = any(count > 0 for count in counts.values())
    score = 1.0 if not ai_matched else 0.0
    
    if pred_name:
        # For optimization, provide score and feedback
        detected_patterns = [name for name, count in counts.items() if count > 0]
        patterns_text = ", ".join(detected_patterns) if detected_patterns else "None"
        total_matches = sum(counts.values())
        feedback = f"Detected AI patterns: {patterns_text} ({total_matches} hits)" 
        return dspy.Prediction(score=score, feedback=feedback)
    else:
        # For evaluation, return score
        return score

# Test with an example from your dataset
example = dataset[0]
response = program(ai_text=example.ai_text)  
result = ai_match(gold=example, pred=response, pred_name="humanize")

print(f"\n{'='*60}")
print(f"Example:")
print(f"{'='*60}")
print(f"Original:\n{example.ai_text}")
print(f"{'-'*60}")
print(f"Humanized:\n{response.human_text}")
print(f"{'='*60}\n")
print(f"Score: {result.score}")
print(f"Feedback: {result.feedback}")


In [ ]:
# 6. Establish a baseline  
evaluate = dspy.Evaluate(
    devset=dataset,
    metric=ai_match,
    num_threads=4,
    display_table=True,
    display_progress=True,
)

print("Evaluating baseline AI detector...")
evaluate(program)

In [ ]:
# 7. Compile your program

# Use a more capable model for optimization
smart_lm = dspy.LM(model='openai/gpt-5.5', api_key=os.getenv("OPENAI_API_KEY"), temperature=1, max_tokens=32000)

optimizer = dspy.GEPA(
    metric=exact_match,
    max_full_evals=5,  # Increase for better results
    num_threads=4,
    track_stats=True,
    use_merge=False,
    reflection_lm=smart_lm  # GPT-5.5 analyzes patterns in AI vs human text
)

print("Optimizing AI detector with GEPA...")
optimized_program = optimizer.compile(
    program,
    trainset=trainset,
    valset=testset[:20],  # Use smaller validation set for faster optimization
)

print("\nEvaluating optimized AI detector...")
optimized_score = evaluate(optimized_program)
print(f"Optimized Accuracy: {optimized_score:.1%}")

In [ ]:
# 8. Test and iterate with challenging edge cases
# Test the optimized program on tricky examples that might fool basic detectors

test_texts = [
    # Tricky AI examples (less obvious patterns)
    "The solution addresses key concerns while maintaining flexibility for future iterations.",
    "This represents a significant step forward in our understanding of the problem space.",
    
    # Tricky human examples (more formal than usual)
    "The presentation went well, though a few technical issues came up during the demo.",
    "I reviewed the proposal and think it needs some work before we move forward.",
    
    # Edge cases - could be either
    "The results demonstrate clear improvements across all metrics.",
    "This approach offers several advantages over traditional methods.",
]

print("Testing optimized AI detector on edge cases:\n")
for text in test_texts:
    result = optimized_program(text=text)
    print(f"Text: {text[:80]}...")
    print(f"Predicted: {result.classification} (confidence: {result.confidence:.2f})\n")

In [ ]:
# 9. Analyze AI tells and patterns
# Let's examine what patterns the model has learned to identify

ai_tells = [
    # Common AI vocabulary and phrases
    ("delve", "The report will delve into the underlying factors contributing to this phenomenon."),
    ("tapestry", "This creates a rich tapestry of interconnected elements."),
    ("moreover", "Moreover, the implications extend beyond the immediate context."),
    ("whilst", "The team maintained focus whilst navigating complex challenges."),
    ("paramount", "It's of paramount importance that we address these concerns."),
    ("myriad", "The solution addresses myriad challenges facing the industry."),
    
    # Em/en dash patterns
    ("em_dash", "The results—which exceeded all expectations—demonstrate clear success."),
    ("en_dash", "The 2023–2024 fiscal year showed remarkable growth."),
]

human_patterns = [
    # Casual language  
    ("contractions", "I can't believe we're doing this again"),
    ("internet_speak", "tbh this whole thing is kinda sus"),
    ("informal", "gonna grab lunch, anyone wanna join?"),
    ("typos", "just realizd i forgot my laptop at home"),
    ("emotions", "ugh why does this always happen to me"),
    ("ellipsis", "not sure about this one..."),
]

print("Testing known AI tells:\n")
for name, text in ai_tells:
    result = optimized_program(text=text)
    print(f"{name:12} | {result.classification:5} ({result.confidence:.2f}) | {text[:50]}...")

print("\n" + "="*80 + "\n")
print("Testing human patterns:\n")
for name, text in human_patterns:
    result = optimized_program(text=text)
    print(f"{name:12} | {result.classification:5} ({result.confidence:.2f}) | {text[:50]}...")


In [ ]:
# 10. Summary and insights

print("="*80)
print("AI TEXT DETECTION SUMMARY")
print("="*80)
print(f"\nBaseline Accuracy: {baseline_score:.1%}")
print(f"Optimized Accuracy: {optimized_score:.1%}")
print(f"Improvement: {(optimized_score - baseline_score):.1%}")

print("\n" + "="*80)
print("KEY PATTERNS IDENTIFIED")
print("="*80)

print("\nCommon AI Tells:")
print("- Em/en dashes (— and –) instead of regular hyphens")
print("- Formal vocabulary: 'delve', 'tapestry', 'myriad', 'whilst', 'paramount'")
print("- Complex sentence structures with multiple clauses")
print("- Abstract concepts and business jargon")
print("- Consistent formatting and perfect grammar")

print("\nCommon Human Patterns:")
print("- Contractions and informal language")
print("- Typos and casual capitalization")
print("- Internet slang and abbreviations (lol, tbh, ngl)")
print("- Emotional expressions and personal anecdotes")
print("- Inconsistent punctuation and ellipses (...)")

print("\nNote: These patterns are statistical tendencies, not absolute rules.")
print("Advanced AI models can mimic human writing, and humans can write formally.")
